# Adversarial Training (Option B) with ART for 5 sklearn Models

This notebook implements a **repeatable adversarial-training loop** for multiple sklearn-style classifiers using the **Adversarial Robustness Toolbox (ART)**.

## Workflow (per model, per round)

1. Train on **clean training data**.
2. Generate **fresh adversarial examples** against the *current* model (from **training data only**, never test).
3. Retrain on a **mix of clean + adversarial** samples with the **correct labels**.
4. Re-attack the **retrained** model and evaluate on a held-out test set:
   - Clean accuracy
   - Adversarial accuracy (attacked test subset)
   - Accuracy drop
5. Repeat for a small number of rounds.

## Constraints satisfied

- **No detectors** are used.
- **No training on test-set adversarial examples.**
- Adversarial examples are **regenerated after each round**.
- Uses **decision-based HSJ** for evaluation on all models.
- Optionally tries **PGD** for adversarial training **only when gradients are supported**; otherwise falls back to HSJ-based adversarial training.
- Designed to be **computationally reasonable**: caps attacked samples, conservative HSJ settings.

> If you already have your 5 models implemented, replace the `build_models()` function and/or the dataset-loading cell.


In [1]:
# Core
import numpy as np
import pandas as pd

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

# ART
from art.estimators.classification import SklearnClassifier
from art.attacks.evasion import HopSkipJump, ProjectedGradientDescent

# Repro
RNG = np.random.default_rng(42)

from MachineLearning.LogReg.LogisticRegression_ML import run_best_model as run_logreg
from MachineLearning.NeuralNetworks.NeuralNet_ML import run_best_model as run_neuralnet
from MachineLearning.RandomForest.RandomForest_ML import run_best_model as run_randomforest
from MachineLearning.SVM.SVM_ML import run_best_model as run_svm
from MachineLearning.XGBoost.XGBoost_ML import run_best_model as run_xgboost

c:\Users\janst\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\janst\AppData\Local\Programs\Python\Python310\lib\site-packages\art\estimators\certification\__init__.py:30: UserWarning: PyTorch not found. Not importing DeepZ or Interval Bound Propagation functionality
  warnings.warn("PyTorch not found. Not importing DeepZ or Interval Bound Propagation functionality")


In [2]:
# === Load / prepare your dataset ===
# Replace this block with your actual data loading.
#
# Expected:
#   X_train, y_train, X_test, y_test
#   X_*: float32 arrays shaped (n_samples, n_features)
#   y_*: int labels shaped (n_samples,) with values 0..K-1
#
# IMPORTANT: We will ONLY generate adversarial examples from TRAIN (never test).

import os

csv_path = "/CSVs/dataset.csv"
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)

    label_col = "anomaly"
    drop_cols = {"anomaly", "train", "channel", "segment"}
    feature_cols = [c for c in df.columns if c not in drop_cols]

    if "train" in df.columns:
        train_df = df[df["train"] == 1].copy()
        test_df  = df[df["train"] == 0].copy()

        X_train = train_df[feature_cols].to_numpy(dtype=np.float32)
        y_train = train_df[label_col].to_numpy(dtype=int)

        X_test  = test_df[feature_cols].to_numpy(dtype=np.float32)
        y_test  = test_df[label_col].to_numpy(dtype=int)
    else:
        X = df[feature_cols].to_numpy(dtype=np.float32)
        y = df[label_col].to_numpy(dtype=int)
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
else:
    # Fallback synthetic example if no CSV is present
    from sklearn.datasets import make_classification
    X, y = make_classification(
        n_samples=4000,
        n_features=30,
        n_informative=15,
        n_redundant=5,
        n_classes=2,
        random_state=42
    )
    X = X.astype(np.float32)
    y = y.astype(int)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

# Ensure labels are 0..K-1
_, y_train = np.unique(y_train, return_inverse=True)
_, y_test = np.unique(y_test, return_inverse=True)

print("Train:", X_train.shape, y_train.shape)
print("Test :", X_test.shape,  y_test.shape)
print("Num classes:", len(np.unique(y_train)))


Train: (3200, 30) (3200,)
Test : (800, 30) (800,)
Num classes: 2


In [3]:
# === Define / register your 5 models ===
# Replace these with YOUR existing models if you already have them.
#
# IMPORTANT: Adversarial training refits models multiple rounds, so we define model factories
# (functions that return a fresh sklearn estimator).

def build_models(random_state: int = 42):

    def _wrap(fn):
        def factory():
            try:
                ret = fn(random_state=random_state)
            except TypeError:
                ret = fn()
            # If returned object is already an estimator, return it
            if hasattr(ret, "fit"):
                return ret
            # If a tuple/list was returned (e.g., (estimator, ...)), find first estimator-like entry
            if isinstance(ret, (tuple, list)):
                for item in ret:
                    if hasattr(item, "fit"):
                        return item
            # Nothing usable found — raise helpful error
            raise TypeError(
                f"Wrapped model factory {getattr(fn, '__name__', repr(fn))} "
                f"returned {type(ret)!r} without 'fit' method. "
                "If your run_best_model returns (estimator, ...), ensure the estimator is first or adapt the wrapper."
            )
        return factory

    return {
        "LogReg": _wrap(run_logreg),
        "SVM-RBF": _wrap(run_svm),
        "RandomForest": _wrap(run_randomforest),
        "XGBoost": _wrap(run_xgboost),
        "MLP": _wrap(run_neuralnet),
    }

model_factories = build_models()
list(model_factories.keys())


['LogReg', 'SVM-RBF', 'RandomForest', 'XGBoost', 'MLP']

In [4]:
# === ART helpers ===

def make_art_sklearn_classifier(model, X_ref: np.ndarray) -> SklearnClassifier:
    # Wrap an sklearn model for ART. clip_values should match the expected input range.
    x_min = float(np.min(X_ref))
    x_max = float(np.max(X_ref))
    clip_values = (x_min, x_max)
    return SklearnClassifier(model=model, clip_values=clip_values)

def predict_labels(art_clf: SklearnClassifier, X: np.ndarray) -> np.ndarray:
    # ART predict may return probabilities (n, K). Convert to labels.
    preds = np.asarray(art_clf.predict(X))
    if preds.ndim == 1:
        return preds.astype(int)
    return np.argmax(preds, axis=1)

def sample_subset(X: np.ndarray, y: np.ndarray, n: int, rng=RNG):
    if n >= len(X):
        return X, y
    idx = rng.choice(len(X), size=n, replace=False)
    return X[idx], y[idx]

def attack_success_rate(y_true: np.ndarray, y_pred_clean: np.ndarray, y_pred_adv: np.ndarray) -> float:
    # ASR among points that were correctly classified on clean inputs.
    mask = (y_pred_clean == y_true)
    if mask.sum() == 0:
        return float("nan")
    return float((y_pred_adv[mask] != y_true[mask]).mean())


In [5]:
# === Attacks configuration ===
# HSJ: decision-based; works broadly but can be expensive.
HSJ_EVAL_KWARGS = dict(
    max_iter=20,
    max_eval=5000,
    init_eval=50,
    init_size=10,
    targeted=False,
    norm=2,
)

# For adversarial training we keep HSJ lighter than eval.
HSJ_TRAIN_KWARGS = dict(
    max_iter=10,
    max_eval=2000,
    init_eval=25,
    init_size=10,
    targeted=False,
    norm=2,
)

# PGD: will only be used if loss gradients are supported (often not for sklearn).
PGD_TRAIN_KWARGS = dict(
    eps=0.2,          # adjust to your feature scaling
    eps_step=0.05,
    max_iter=20,
    targeted=False,
    num_random_init=1,
)


In [6]:
# === Adversarial example generation ===

def try_generate_pgd(art_clf: SklearnClassifier, X: np.ndarray, y: np.ndarray):
    # Try PGD if supported. Many sklearn estimators will NOT support gradients.
    try:
        pgd = ProjectedGradientDescent(estimator=art_clf, **PGD_TRAIN_KWARGS)
        X_adv = pgd.generate(x=X, y=y)
        return np.asarray(X_adv, dtype=np.float32)
    except Exception:
        return None

def generate_hsj(art_clf: SklearnClassifier, X: np.ndarray, y: np.ndarray, *, train_mode: bool) -> np.ndarray:
    kwargs = HSJ_TRAIN_KWARGS if train_mode else HSJ_EVAL_KWARGS
    hsj = HopSkipJump(classifier=art_clf, **kwargs)
    X_adv = hsj.generate(x=X, y=y)
    return np.asarray(X_adv, dtype=np.float32)

def generate_adversarials(art_clf: SklearnClassifier, X: np.ndarray, y: np.ndarray, *, prefer_pgd: bool, train_mode: bool):
    # Generate adversarial examples for (X,y).
    # - If prefer_pgd: try PGD first; if it fails, fall back to HSJ.
    if prefer_pgd:
        X_adv = try_generate_pgd(art_clf, X, y)
        if X_adv is not None:
            return X_adv, "PGD"
    return generate_hsj(art_clf, X, y, train_mode=train_mode), "HSJ"


In [7]:
# === Adversarial training loop (Option B) ===

def adversarial_training_loop(
    model_name: str,
    model_factory,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    *,
    rounds: int = 2,
    train_adv_samples: int = 300,   # how many TRAIN points to adversarially augment per round
    eval_adv_samples: int = 200,    # how many TEST points to attack for evaluation
    prefer_pgd_for_training: bool = True,
    rng=RNG,
):
    # Start with clean training set (we will augment this over rounds)
    X_tr = np.asarray(X_train, dtype=np.float32)
    y_tr = np.asarray(y_train, dtype=int)

    history = []

    for r in range(rounds + 1):
        # 1) Train current model
        model = model_factory()
        model.fit(X_tr, y_tr)

        # Wrap for ART
        art_clf = make_art_sklearn_classifier(model, X_tr)

        # 2) Clean test accuracy (full test set)
        y_pred_clean_full = predict_labels(art_clf, X_test)
        clean_acc = accuracy_score(y_test, y_pred_clean_full)

        # 3) Adversarial test accuracy on a subset (never used for training)
        X_eval, y_eval = sample_subset(X_test, y_test, eval_adv_samples, rng=rng)
        X_adv_eval, eval_attack_used = generate_adversarials(
            art_clf, X_eval, y_eval,
            prefer_pgd=False,   # evaluation: HSJ
            train_mode=False
        )
        y_pred_clean_eval = predict_labels(art_clf, X_eval)
        y_pred_adv_eval = predict_labels(art_clf, X_adv_eval)

        adv_acc = accuracy_score(y_eval, y_pred_adv_eval)
        acc_drop = clean_acc - adv_acc
        asr = attack_success_rate(y_eval, y_pred_clean_eval, y_pred_adv_eval)

        history.append({
            "Model": model_name,
            "Round": r,
            "TrainSetSize": int(len(X_tr)),
            "EvalAttack": eval_attack_used,
            "CleanAcc": float(clean_acc),
            "AdvAcc": float(adv_acc),
            "AccDrop": float(acc_drop),
            "AttackSuccessRate": float(asr),
        })

        print(f"[{model_name}] round={r} clean={clean_acc:.4f} adv={adv_acc:.4f} drop={acc_drop:.4f} ASR={asr:.4f}")

        if r == rounds:
            break

        # 4) Generate FRESH adversarial examples from TRAIN subset only
        X_sub, y_sub = sample_subset(X_train, y_train, train_adv_samples, rng=rng)

        X_adv_train, train_attack_used = generate_adversarials(
            art_clf, X_sub, y_sub,
            prefer_pgd=prefer_pgd_for_training,
            train_mode=True
        )

        # 5) Retrain next round on clean+adv with correct labels
        X_tr = np.vstack([X_tr, X_adv_train]).astype(np.float32)
        y_tr = np.concatenate([y_tr, y_sub]).astype(int)

        print(f"    + augmented {len(X_adv_train)} adversarials using {train_attack_used}; new train size={len(X_tr)}")

    return pd.DataFrame(history)


In [ ]:
# === Run adversarial training across all models ===

all_hist = []
for name, factory in model_factories.items():
    df_hist = adversarial_training_loop(
        model_name=name,
        model_factory=factory,
        X_train=X_train, y_train=y_train,
        X_test=X_test, y_test=y_test,
        rounds=2,

        # FULL DATASET SETTINGS
        train_adv_samples=len(X_train),   # attack ALL train points for augmentation each round
        eval_adv_samples=len(X_test),     # attack ALL test points for adversarial evaluation each round

        prefer_pgd_for_training=True,
    )
    all_hist.append(df_hist)

results_df = pd.concat(all_hist, ignore_index=True)
results_df



Using parameters: {'clf__C': 10, 'clf__class_weight': 'balanced', 'clf__penalty': 'l2', 'clf__solver': 'liblinear'}

Saved last run results to Results/LogRegResults\results_logreg.csv
Saved ROC data to Results/LogRegResults\roc_logreg_clean.csv (AUC = 0.955)
Saved summary (AUC + Confusion Matrix) to Results/LogRegResults\logreg_summary.csv

=== Test Set Classification Report ===
              precision    recall  f1-score   support

           0       0.96      0.93      0.95      1352
           1       0.77      0.86      0.81       347

    accuracy                           0.92      1699
   macro avg       0.86      0.90      0.88      1699
weighted avg       0.92      0.92      0.92      1699


Confusion Matrix:
         Pred 0  Pred 1
True 0    1260      92
True 1      47     300

AUC: 0.955

=== All results and summaries saved successfully ===


HopSkipJump: 100%|██████████| 800/800 [00:48<00:00, 16.44it/s]


[LogReg] round=0 clean=0.7887 adv=0.2125 drop=0.5762 ASR=0.9968


HopSkipJump: 100%|██████████| 3200/3200 [01:38<00:00, 32.50it/s]


    + augmented 3200 adversarials using HSJ; new train size=6400
Using parameters: {'clf__C': 10, 'clf__class_weight': 'balanced', 'clf__penalty': 'l2', 'clf__solver': 'liblinear'}

Saved last run results to Results/LogRegResults\results_logreg.csv
Saved ROC data to Results/LogRegResults\roc_logreg_clean.csv (AUC = 0.955)
Saved summary (AUC + Confusion Matrix) to Results/LogRegResults\logreg_summary.csv

=== Test Set Classification Report ===
              precision    recall  f1-score   support

           0       0.96      0.93      0.95      1352
           1       0.77      0.86      0.81       347

    accuracy                           0.92      1699
   macro avg       0.86      0.90      0.88      1699
weighted avg       0.92      0.92      0.92      1699


Confusion Matrix:
         Pred 0  Pred 1
True 0    1260      92
True 1      47     300

AUC: 0.955

=== All results and summaries saved successfully ===


...

HopSkipJump: 100%|██████████| 3200/3200 [03:56<00:00, 13.52it/s]


    + augmented 3200 adversarials using HSJ; new train size=9600

Saved last run results to Results/SVMResults\results_svm.csv
Saved ROC data to Results/SVMResults\roc_svm_clean.csv (AUC = 0.934)
Saved summary (AUC + Confusion Matrix) to Results/SVMResults\svm_summary.csv

=== SVM Test Set Report ===
              precision    recall  f1-score   support

           0       0.93      0.99      0.96      1520
           1       0.94      0.70      0.80       391

    accuracy                           0.93      1911
   macro avg       0.93      0.84      0.88      1911
weighted avg       0.93      0.93      0.93      1911


Confusion Matrix:
         Pred 0  Pred 1
True 0    1503      17
True 1     117     274

AUC: 0.934

=== All results and summaries saved successfully ===


HopSkipJump: 100%|██████████| 800/800 [04:58<00:00,  2.68it/s]


[SVM-RBF] round=2 clean=0.7788 adv=0.2225 drop=0.5563 ASR=0.9984

Saved last run results to Results/RandomForestResults\results_randomforest.csv
Saved ROC data to Results/RandomForestResults\roc_randomforest_clean.csv (AUC = 0.972)
Saved summary (AUC + Confusion Matrix) to Results/RandomForestResults\randomforest_summary.csv

=== Random Forest Test Set Report ===
              precision    recall  f1-score   support

           0       0.95      0.98      0.97      1352
           1       0.93      0.80      0.86       347

    accuracy                           0.95      1699
   macro avg       0.94      0.89      0.91      1699
weighted avg       0.94      0.95      0.94      1699


Confusion Matrix:
         Pred 0  Pred 1
True 0    1330      22
True 1      71     276

AUC: 0.972

=== All results and summaries saved successfully ===


HopSkipJump:  16%|█▋        | 131/800 [10:10<53:59,  4.84s/it] 

In [ ]:
# === Summary view ===

summary = results_df.pivot_table(
    index=["Model", "Round"],
    values=["CleanAcc", "AdvAcc", "AccDrop", "AttackSuccessRate", "TrainSetSize"],
    aggfunc="mean"
).reset_index()

summary.sort_values(["Model", "Round"])


## Interpreting the metrics

- **CleanAcc**: accuracy on the full held-out test set (clean).
- **AdvAcc**: accuracy on a *subset of the test set* after attack (HSJ). This is a practical robustness estimate without the full computational cost.
- **AccDrop**: `CleanAcc - AdvAcc`. Lower is better.
- **AttackSuccessRate (ASR)**: among test points the model got right on clean inputs, fraction that become wrong under attack. Lower is better.

### Why this is meaningful

- Training uses **only training data** and adversarials generated from it.
- Evaluation attacks the **test set** but does **not** train on those attacks.
- Adversarial examples are regenerated **each round** against the updated model.
- Metrics are tracked per-round, per-model, so you can see whether adversarial training improves robustness (AdvAcc) without destroying clean accuracy.

### Practical knobs

- `train_adv_samples`: how many training points you adversarially augment per round.
- HSJ settings (`HSJ_TRAIN_KWARGS`, `HSJ_EVAL_KWARGS`): increase/decrease for quality vs runtime.
- `rounds`: 1–3 is typical for a course project.

> If HSJ is slow: reduce `train_adv_samples`, `eval_adv_samples`, and/or `max_iter/max_eval`.
